In [3]:
import torch
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd

In [4]:
test_results = pd.read_csv('config.csv', dtype=str)
# to dataframe
test_results = pd.DataFrame(test_results)
# Strip any leading or trailing spaces from the column names
test_results.columns = test_results.columns.str.strip()
test_results = test_results.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

for i in range(len(test_results)):
    seeds = test_results.iloc[i]['seed'].split(' ')
    dataset = test_results.iloc[i]['dataset']
    usage = test_results.iloc[i]['usage']
    model = test_results.iloc[i]['model']
    lr = test_results.iloc[i]['lr']
    snr = test_results.iloc[i]['snr']
    snvr = test_results.iloc[i]['snvr']
    weight_decay = test_results.iloc[i]['weight_decay']
    
    valid_top1_acc_list = []
    valid_top5_acc_list = []
    test_top1_acc_list = []
    test_top5_acc_list = []

    ckpt_path = f'./ckpt/{dataset}_{usage}_{model}_{lr}_{snr}_{snvr}_{weight_decay}.pth'
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path)
    
        for seed in ckpt.keys():
            valid_top1_acc_list.append(ckpt[seed]['best_epoch']['valid_top1_acc'])
            valid_top5_acc_list.append(ckpt[seed]['best_epoch']['valid_top5_acc'])
            test_top1_acc_list.append(ckpt[seed]['best_epoch']['test_top1_acc'])
            test_top5_acc_list.append(ckpt[seed]['best_epoch']['test_top5_acc'])
    
    test_results.at[i, 'seed'] = f'{len(valid_top1_acc_list)}/{len(seeds)}'
    if len(valid_top1_acc_list) == 0: continue
    test_results.at[i, 'valid_acc@1'] = f'{np.mean(valid_top1_acc_list):.4f}'
    test_results.at[i, 'valid_acc@5'] = f'{np.mean(valid_top5_acc_list):.4f}'
    test_results.at[i, 'test_acc@1'] = f'{np.mean(test_top1_acc_list):.4f}'
    test_results.at[i, 'test_acc@5'] = f'{np.mean(test_top5_acc_list):.4f}'

test_results = test_results.drop(columns=['epochs','warm_up','batch_size'])
print(test_results.to_string())

   seed   dataset usage      model    lr    snr snvr weight_decay  optim
0   0/1  cifar100   0.2  resnet_18  1e-3    0.0  0.0         5e-3   adam
1   0/1  cifar100   0.2  resnet_18  1e-3    0.0  0.0         5e-4   adam
2   0/1  cifar100   0.2  resnet_18  1e-3    0.0  0.0         5e-5   adam
3   0/1  cifar100   0.2  resnet_18  1e-3    1.0  0.0          0.0   adam
4   0/1  cifar100   0.2  resnet_18  1e-3    0.1  0.0          0.0   adam
5   0/1  cifar100   0.2  resnet_18  1e-3   0.01  0.0          0.0   adam
6   0/1  cifar100   0.2  resnet_18  1e-3  0.001  0.0          0.0   adam
7   0/5  cifar100   1.0  resnet_18  1e-3    0.0  0.0         1e-2  adamw
8   0/5  cifar100   1.0  resnet_18  1e-3   1e-4  0.0         1e-2  adamw
9   0/5  cifar100   1.0  resnet_18  1e-3   1e-5  0.0         1e-2  adamw
10  0/5  cifar100   1.0  resnet_18  1e-3   1e-4  0.1         1e-2  adamw
11  0/1  cifar100   1.0  resnet_18  1e-3   1e-4  1.0         1e-2  adamw
12  0/5  cifar100   0.5  resnet_18  1e-3    0.0  0.